## 1. Initializing the project

### 1.1 Initial imports from needed packages

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import osmium
import time
from collections import Counter
import numpy as np
import shapely


### 1.2 Saving the file paths and checking their size

In [2]:
DATA_DIR = Path("data")
OSH_PATH = DATA_DIR / "history_file.osh.pbf"
AOI_PATH = DATA_DIR / "aoi.geojson"   # or .shp, .gpkg
OSH_PATH_CLIPPED = DATA_DIR / "clipped_history_file.osh.pbf"
HISTORY_PATH = DATA_DIR / "history_table.parquet"

print(f"{OSH_PATH.stat().st_size / 1e6:.2f} MB")
print(f"{AOI_PATH.stat().st_size / 1e3:.2f} KB")

141.13 MB
0.43 KB


### 1.3 Read the AOI and set the projection

In [ ]:
aoi = gpd.read_file(AOI_PATH)

print(aoi.crs, len(aoi), aoi.geom_type.unique())

if aoi.crs is None:
    raise ValueError("AOI has no CRS: please set it explicitly")
aoi = aoi.to_crs(4326)


EPSG:4326 1 <ArrowStringArray>
['MultiPolygon']
Length: 1, dtype: str


### 1.4 Inspect PBF file

In [4]:
reader = osmium.io.Reader(str(OSH_PATH), osmium.osm.osm_entity_bits.NOTHING)
header = reader.header()

print("History file:", header.has_multiple_object_versions)
print("Bounding box:", header.box())
print("Generator:", header.get("generator"))
print("Replication timestamp:", header.get("osmosis_replication_timestamp"))
reader.close()

History file: True
Bounding box: (35.2770970/32.3066000 42.3838300/37.3223940)
Generator: osmium/1.16.0
Replication timestamp: 


#### 1.4.1 Stream the history file and describe it

It is necessary to define a class to use pyosmium SimpleHandler and read the elemetns in the history file in order to make counts and preview the amount of info that comes in the pbf file

In [ ]:
class HistorySummary(osmium.SimpleHandler): ##Defining subclass because pyosmium requires it.
    def __init__(self): 
        super().__init__()
        self.versions = Counter()        # all versions, per type
        self.objects = Counter()         # distinct objects, per type
        self.deletions = Counter()       # versions where the object was deleted
        self.edits_per_year = Counter()
        self.users = set()
        self.changesets = set()
        self.first_ts = None
        self.last_ts = None
        self._last_id = {}               # last ID seen, per type

    def _process(self, otype, o):
        # Distinct-object count, relying on (type, id, version) sort order
        prev = self._last_id.get(otype)
        if prev is not None and o.id < prev:
            raise RuntimeError(f"File not sorted: {otype} {o.id} after {prev}")
        if o.id != prev:
            self.objects[otype] += 1
            self._last_id[otype] = o.id

        self.versions[otype] += 1
        if not o.visible:
            self.deletions[otype] += 1

        ts = o.timestamp
        if self.first_ts is None or ts < self.first_ts:
            self.first_ts = ts
        if self.last_ts is None or ts > self.last_ts:
            self.last_ts = ts
        self.edits_per_year[ts.year] += 1

        self.users.add(o.uid)
        self.changesets.add(o.changeset)

    def node(self, n):
        self._process("node", n)

    def way(self, w):
        self._process("way", w)

    def relation(self, r):
        self._process("relation", r)


summary = HistorySummary()
t0 = time.perf_counter()
summary.apply_file(str(OSH_PATH))
print(f"Done in {time.perf_counter() - t0:.1f} s")

Done in 140.3 s


Printing the summary of the information in the whole pbf file

In [7]:
print("First edit:", summary.first_ts)
print("Last edit: ", summary.last_ts)
print()
for t in ("node", "way", "relation"):
    print(f"{t:9} objects={summary.objects[t]:>10,}  "
          f"versions={summary.versions[t]:>10,}  "
          f"deletions={summary.deletions[t]:>9,}")
print()
print("Distinct users: ", f"{len(summary.users):,}")
print("Distinct changesets:", f"{len(summary.changesets):,}")
print()
print(dict(sorted(summary.edits_per_year.items())))

First edit: 2006-05-15 01:04:26+00:00
Last edit:  2026-09-20 23:12:15+00:00

node      objects=13,900,091  versions=17,694,724  deletions=  964,630
way       objects= 1,986,632  versions= 2,627,127  deletions=  119,553
relation  objects=     4,187  versions=    56,215  deletions=      753

Distinct users:  25,045
Distinct changesets: 251,091

{2006: 171, 2007: 27232, 2008: 152047, 2009: 209031, 2010: 83833, 2011: 309493, 2012: 711170, 2013: 571949, 2014: 404166, 2015: 605654, 2016: 696982, 2017: 664121, 2018: 309627, 2019: 555433, 2020: 575386, 2021: 1693430, 2022: 527247, 2023: 8143854, 2024: 908711, 2025: 1398886, 2026: 1829643}


### 1.5 Clip the history file to the AOI

Merge the AOI into one geometry and get its bounding box

In [ ]:
aoi_geom = aoi.union_all()          # one MultiPolygon geometry
shapely.prepare(aoi_geom)           # Create spatial index to speeds up processes 
minx, miny, maxx, maxy = aoi_geom.bounds
print(aoi_geom.bounds)

(35.896452, 32.72490980936, 36.177534086, 33.41195703719)


#### 1.5.1 Pre-filter nodes by the AOI bounding box

Collect the ID and coordinates of every visible node version with a valid location that falls inside the AOI’s bounding box. A node can appear more than once (one entry per version). 

In [7]:
ids, lons, lats = [], [], []

t0 = time.perf_counter()
for n in osmium.FileProcessor(str(OSH_PATH), osmium.osm.NODE):
    if not n.visible:               # deleted versions have no location
        continue
    loc = n.location
    if not loc.valid():
        continue
    lon, lat = loc.lon, loc.lat
    if minx <= lon <= maxx and miny <= lat <= maxy:
        ids.append(n.id)
        lons.append(lon)
        lats.append(lat)
print(f"{len(ids):,} node versions in bbox, {time.perf_counter() - t0:.0f} s")

180,974 node versions in bbox, 80 s


Keep the IDs of the nodes whose coordinates fall inside the AOI polygon, without repeats, and save them in node_ids.

In [ ]:
ids = np.asarray(ids, dtype=np.int64)
inside = shapely.intersects_xy(aoi_geom, np.asarray(lons), np.asarray(lats))

node_ids = set(np.unique(ids[inside]).tolist())
print(f"{inside.sum():,} node versions inside AOI")
print(f"{len(node_ids):,} distinct nodes selected")

del lons, lats, inside              # IDs are enough

180,973 node versions inside AOI
160,236 distinct nodes selected


#### 1.5.2 Select the ways

Get the IDs of all ways that, in any version, use at least one node inside the AOI, and save them in way_ids.

In [9]:
way_ids = set()

t0 = time.perf_counter()
for w in osmium.FileProcessor(str(OSH_PATH), osmium.osm.WAY):
    if any(nd.ref in node_ids for nd in w.nodes):
        way_ids.add(w.id)
print(f"{len(way_ids):,} ways selected, {time.perf_counter() - t0:.0f} s")

19,676 ways selected, 57 s


**Complete the ways:** Read all ways again and, for those in way_ids, collect the IDs of every node they use (in any version). Keep only the nodes not already in node_ids. These are the nodes outside the AOI that the selected ways need to be complete.

In [10]:
extra_node_ids = set()

t0 = time.perf_counter()
for w in osmium.FileProcessor(str(OSH_PATH), osmium.osm.WAY):
    if w.id in way_ids:
        extra_node_ids.update(nd.ref for nd in w.nodes)

extra_node_ids -= node_ids          # keep only the new ones
print(f"{len(extra_node_ids):,} extra nodes to complete ways, "
      f"{time.perf_counter() - t0:.0f} s")

16,147 extra nodes to complete ways, 9 s


#### 1.5.3 Select the relations

Some features aren’t a single node or way, but a relation: a group of members, each with a role. 

Get the IDs of all relations that, in any version, have at least one selected node or way as a member, and save them in relation_ids. Also count the relation versions by type.

In [11]:
relation_ids = set()
relation_kinds = Counter()          # kinds of relation versions found

t0 = time.perf_counter()
for r in osmium.FileProcessor(str(OSH_PATH), osmium.osm.RELATION):
    if any((m.type == "n" and m.ref in node_ids) or
           (m.type == "w" and m.ref in way_ids)
           for m in r.members):
        relation_ids.add(r.id)
        relation_kinds[r.tags.get("type", "no type")] += 1
print(f"{len(relation_ids):,} relations selected, {time.perf_counter() - t0:.0f} s")
print(relation_kinds.most_common())

84 relations selected, 25 s
[('boundary', 1134), ('route', 34), ('multipolygon', 31), ('waterway', 16), ('no type', 12), ('linestring', 6), ('border', 1), ('restriction', 1)]


#### 1.5.4 Keep only the useful relations and complete them

**Filter the relations:** Keep only the relations that are multipolygons (in any version) and drop the rest.

In [ ]:
mp_relation_ids = set()     # multipolygon relations is the only one i have to save
extra_way_ids = set()       # the member ways i don't have yet and i need to complete them

t0 = time.perf_counter()
for r in osmium.FileProcessor(str(OSH_PATH), osmium.osm.RELATION):
    if r.id in relation_ids and r.tags.get("type") == "multipolygon":
        mp_relation_ids.add(r.id)
        for m in r.members:
            if m.type == "w": ## brings just the ways
                extra_way_ids.add(m.ref)

extra_way_ids -= way_ids    # keep only the new ones
print(f"{len(mp_relation_ids):,} multipolygon relations kept, "
      f"{len(extra_way_ids):,} extra ways needed, "
      f"{time.perf_counter() - t0:.0f} s")

19 multipolygon relations kept, 6 extra ways needed, 1 s


#### 1.5.5 Nodes of the extra ways
Collect the nodes of the extra ways

In [ ]:
new_node_ids = set()                 # nodes used by the extra ways

t0 = time.perf_counter()
for w in osmium.FileProcessor(str(OSH_PATH), osmium.osm.WAY):
    if w.id in extra_way_ids:
        new_node_ids.update(nd.ref for nd in w.nodes)

new_node_ids -= node_ids             # already have: inside the AOI
new_node_ids -= extra_node_ids       # already have: from 1.5.2
extra_node_ids.update(new_node_ids)  # add JUST the new ones

print(f"{len(new_node_ids):,} new nodes added, "
      f"{len(extra_node_ids):,} extra nodes in total, "
      f"{time.perf_counter() - t0:.0f} s")

804 new nodes added, 16,951 extra nodes in total, 7 s


#### 1.5.6 Save the entities clipped in a new history file

In [ ]:
keep = {
    "n": node_ids | extra_node_ids,     # nodes to keep
    "w": way_ids | extra_way_ids,       # ways to keep
    "r": mp_relation_ids,               # relations to keep
}
written = Counter()                      # versions written, per type

t0 = time.perf_counter()
with osmium.SimpleWriter(str(OSH_PATH_CLIPPED), overwrite=True) as writer:
    for row in osmium.FileProcessor(str(OSH_PATH)):
        kind = row.type_str()              # "n", "w" or "r"
        if row.id in keep[kind]:
            writer.add(row)
            written[kind] += 1

print(f" Elements written: {dict(written)}")
print(f"{OSH_PATH_CLIPPED.stat().st_size / 1e6:.1f} MB written "
      f"in {time.perf_counter() - t0:.0f} s")


 Elements written: {'n': 212130, 'w': 27480, 'r': 34}
1.8 MB written in 62 s


Printing the summary of the information in the clipped pbf file

In [18]:
summary_clipped = HistorySummary()
t0 = time.perf_counter()
summary_clipped.apply_file(str(OSH_PATH_CLIPPED))
print(f"Done in {time.perf_counter() - t0:.1f} s")

print("First edit:", summary_clipped.first_ts)
print("Last edit: ", summary_clipped.last_ts)
print()
for t in ("node", "way", "relation"):
    print(f"{t:9} objects={summary_clipped.objects[t]:>10,}  "
          f"versions={summary_clipped.versions[t]:>10,}  "
          f"deletions={summary_clipped.deletions[t]:>9,}")
print()
print("Distinct users: ", f"{len(summary_clipped.users):,}")
print("Distinct changesets:", f"{len(summary_clipped.changesets):,}")
print()
print(dict(sorted(summary_clipped.edits_per_year.items())))

Done in 1.6 s
First edit: 2007-08-11 19:44:33+00:00
Last edit:  2026-09-19 13:57:24+00:00

node      objects=   177,187  versions=   212,130  deletions=    9,316
way       objects=    19,682  versions=    27,480  deletions=      580
relation  objects=        19  versions=        34  deletions=        3

Distinct users:  368
Distinct changesets: 2,529

{2007: 17, 2008: 2698, 2009: 2498, 2010: 613, 2011: 16281, 2012: 28267, 2013: 15260, 2014: 1788, 2015: 15311, 2016: 4301, 2017: 1890, 2018: 13142, 2019: 3807, 2020: 2383, 2021: 4128, 2022: 763, 2023: 4359, 2024: 9215, 2025: 8255, 2026: 104668}


## 2. PROCESS THE PBF FILE CLIPPED SO THE DATA IS ACCESSIBLE TO MAKE ANALYSIS

### 2.1 Feature classes

**Tag values that define the classes to be assessed:**

1. Roads (highway=* values below)
2. POIs (amenity=* values below). 
3. Buildings are any feature with a building tag, except building=no. More values can be added here if needed.

In [4]:
ROAD_TYPES = {
    "motorway", "motorway_link", "trunk", "trunk_link",
    "primary", "primary_link", "secondary", "secondary_link",
    "tertiary", "tertiary_link", "unclassified", "residential",
    "living_street", "service", "track", "pedestrian", "footway",
}
POI_TYPES = {"hospital", "clinic", "place_of_worship"}

### 2.2 Define the classification function 

**Classify a version:** The function receives one version (row) of the history file and returns the set of classes it belongs to, based on its tags: road, building, poi, more than one (e.g. a mosque that is a building and a POI), or none.

In [5]:
def classify(element):
    """Return the set of feature classes of one version (can be empty, e.g:  deleted objects)."""
    tags = element.tags
    classes = set()
    building_value = tags.get("building") 

    if element.is_way() and tags.get("highway") in ROAD_TYPES:
        classes.add("road")
    if not element.is_node() and building_value is not None and building_value != "no":
        classes.add("building")
    if tags.get("amenity") in POI_TYPES:
        classes.add("poi")
    return classes


### 2.3 Defining the history table 

**Build and define the history table:** Read the clipped history file and keep one row per version of every road, building and POI. A deleted version has no tags, so it takes the class of the previous version of the same object. A version is also kept when the object leaves the classes (e.g. a building re-tagged as farmland) to keep those deletions or counts. The table is saved as history_table.parquet and used by the later notebooks.

| Column	| Meaning |
| -- | -- | 
| feature_id |	unique key: type letter + ID (e.g. w500) |
|type, id, version |	OSM type (n/w/r), ID and version number |
| visible |	False if this version deleted the object |
|timestamp, uid, user, changeset	| when, by whom and in which changeset the version was made |
| is_road, is_building, is_poi	| classes of this version |
| highway, building, amenity, name	| values of these tags (empty if absent) |
| n_tags	| number of tags |
| n_nodes	| number of nodes (ways only)|

In [20]:
rows = []           # one dictionary per version (per row)
prev_key = None     # previous row: which object
prev_cls = set()    #previous row: which class

t0 = time.perf_counter()
for o in osmium.FileProcessor(str(OSH_PATH_CLIPPED)):
    
    key = (o.type_str(), o.id)
    was = prev_cls if key == prev_key else set()  # class of the previous version

    if o.visible:
        cls = classify(o)
    else:
        cls = was                  # deleted: no tags, so inherit the class

    if len(cls) > 0 or len(was) > 0:
        rows.append({
            "feature_id": f"{o.type_str()}{o.id}",
            "type": o.type_str(),
            "id": o.id,
            "version": o.version,
            "visible": o.visible,
            "timestamp": o.timestamp,
            "uid": o.uid,
            "user": o.user,
            "changeset": o.changeset,
            "is_road": "road" in cls,
            "is_building": "building" in cls,
            "is_poi": "poi" in cls,
            "highway": o.tags.get("highway"),
            "building": o.tags.get("building"),
            "amenity": o.tags.get("amenity"),
            "name": o.tags.get("name"),
            "n_tags": len(o.tags),
            "n_nodes": len(o.nodes) if o.is_way() else None,
        })

    prev_key, prev_cls = key, cls

history = pd.DataFrame(rows)
history.to_parquet(HISTORY_PATH, index=False)

print(f"{len(history):,} rows, {time.perf_counter() - t0:.1f} s")
print(history[["is_road", "is_building", "is_poi"]].sum())
history.head(3)

23,103 rows, 1.7 s
is_road        12119
is_building    10894
is_poi           100
dtype: int64


,feature_id,type,id,version,visible,timestamp,uid,user,changeset,is_road,is_building,is_poi,highway,building,amenity,name,n_tags,n_nodes
0,n316007195,n,316007195,1,True,2008-12-01 06:59:50+00:00,49904,jonnymaserati,6931,False,False,True,NaN,NaN,place_of_worship,Abbey of St Paul the Messenger,3,NaN
1,n316007195,n,316007195,2,True,2008-12-04 10:18:06+00:00,49904,jonnymaserati,95532,False,False,True,NaN,NaN,place_of_worship,Abbey of St Paul the Messenger,4,NaN
2,n316007195,n,316007195,3,True,2008-12-19 12:27:28+00:00,49904,jonnymaserati,483695,False,False,True,NaN,NaN,place_of_worship,Abbey of St Paul the Messenger,4,NaN


### 2.4 Exploratory Data Analysis

In order to know more about the information that was extracted from the PBF file, the following questions are gonna be answer:

1. **Structure:** How many rows there are? and, are the column types right? 
2. **Features vs versions:** How many features per class and how many are alive today 
3. **Domains:** Which values appear in highway, building, amenity (this was already filtered according the tags explicitly defined in some cells above, but the total amount is gonna be checked)
4. **Time:** How many edits were per year and per class 
5. **Who:** How many users and changesets made those edits (Were there any bulk import or campaign?)
6. **Sanity checks:** Checking duplicates in feature-version and nodes

#### 2.4.1. Structure 

How many rows there are? and, are the column types right? 

In [ ]:
print(f"Size of the history table: {history.shape}")
print("")
print(f"Data types in the columns: ")
history.dtypes

Size of the df: (23103, 18)

Data types in the columns: 


feature_id                     str
type                           str
id                           int64
version                      int64
visible                       bool
timestamp      datetime64[us, UTC]
uid                          int64
user                           str
changeset                    int64
is_road                       bool
is_building                   bool
is_poi                        bool
highway                        str
building                       str
amenity                        str
name                           str
n_tags                       int64
n_nodes                    float64
dtype: object

Total features per class

In [18]:
print(f"Total of features: {history['feature_id'].nunique()}")

for col in ["is_road", "is_building", "is_poi"]:
    n = history[history[col]]["feature_id"].nunique()
    print(f"{col}: {n:,} features")

Total of features: 17159
is_road: 6,586 features
is_building: 10,523 features
is_poi: 68 features


#### 2.4.2 Features vs versions

How many features per class and how many are alive today 

In [21]:
latest = (history
          .sort_values(["feature_id", "version"])           
          .drop_duplicates("feature_id", keep="last"))     

alive = latest[latest["visible"]]                          # those that are visible right now

for col in ["is_road", "is_building", "is_poi"]:
    print(f"{col}: {alive[col].sum():,} alive today")       

is_road: 6,362 alive today
is_building: 10,432 alive today
is_poi: 54 alive today


#### 2.4.3. Domains

Which values appear in highway, building, amenity (this was already filtered according the tags explicitly defined in some cells above, but the total amount is gonna be checked)

In [31]:
alive["building"].value_counts(dropna=False)  

building
yes            9674
NaN            6408
house           471
residential     211
school           40
barn              8
mosque            6
shed              6
industrial        5
apartments        4
greenhouse        4
bunker            2
retail            1
hospital          1
Name: count, dtype: int64

In [28]:
alive["highway"].value_counts(dropna=False)  

highway
NaN               10479
residential        3827
unclassified       1210
track               451
tertiary            314
service             263
secondary           145
living_street        74
primary              58
primary_link         10
secondary_link        7
footway               2
tertiary_link         1
Name: count, dtype: int64

In [29]:
alive["amenity"].value_counts(dropna=False)  

amenity
NaN                 16775
place_of_worship       45
hospital                8
police                  4
school                  3
public_building         3
clinic                  1
townhall                1
library                 1
Name: count, dtype: int64

#### 2.4.4 Time 
How many edits were per year and per class 

In [ ]:
history.groupby(history["timestamp"].dt.year)[["is_road", "is_building", "is_poi"]].sum()

,is_road,is_building,is_poi
timestamp,,,
2008,77,0,3
2009,45,0,2
2010,16,0,0
2011,1623,15,34
2012,1506,0,3
2013,833,0,2
2014,134,5,10
2015,889,9,6
2016,285,3,8


#### 2.2.5 Who
How many users and changesets made those edits (Were there any bulk import or campaign?)

In [51]:
history.groupby(history["timestamp"].dt.year).agg(
    edits=("feature_id", "size"),
    users=("user", "nunique"),
    changesets=("changeset", "nunique"),
)

,edits,users,changesets
timestamp,,,
2008,80,4,13
2009,47,6,16
2010,16,6,6
2011,1672,6,93
2012,1509,11,88
2013,835,9,37
2014,149,20,46
2015,901,17,138
2016,296,16,33


As there's a spike in 2026, how many users and changesets did those edits?

In [ ]:
history2026 = history[history["timestamp"].dt.year == 2026]         # focusing on just 2026

print(f"Edits in 2026: {len(history2026)}")
print(f"Users: {history2026['user'].nunique()}")
print(f"Changesets: {history2026['changeset'].nunique()}")
print("")
print(history2026["user"].value_counts().head(10))                    # top 10 users in 2026

Edits in 2026: 14453
Users: 58
Changesets: 707

user
Samer Aldahhak       7027
Pragya Joshi         1873
noam15A              1139
Laila Khaled          755
Raneem Aldemashki     660
BadainJaranDesert     418
Brawi                 342
Muhammad_khair        322
WanakaLake            308
SamaShamma            285
Name: count, dtype: int64


#### 2.4.6 Sanity checks
Checking duplicates in feature-version 

In [55]:
history.duplicated(subset=["feature_id", "version"], keep=False).sum()

np.int64(0)

Checking roads with less than two nodes

In [60]:
roads_with_errors = alive[(alive["is_road"]) & (alive["n_nodes"] < 2)]
print(len(roads_with_errors))

0


Checking buildings with less than four nodes

In [61]:
buildings_with_errors = alive[(alive["is_building"]) & (alive["n_nodes"] < 4)]
print(len(buildings_with_errors))

0


Checking dates

In [63]:
print(f"Dates from {history['timestamp'].min()} to {history['timestamp'].max()}")

Dates from 2008-06-15 18:49:48+00:00 to 2026-09-19 13:57:24+00:00


#### 2.4.7 Summary and results

- 17,159 features in the three classes: 
  - 6,586 roads
  - 10,523 buildings
  - 68 POIs (18 are both a building and a POI) 
- Features that are "alive" today: 
  - 6,362 roads
  - 10,432 buildings
  - 54 POIs
- POIs lost the most: 14 of 68 no longer exist or left the class
- Aproximately 94% of buildings are tagged only building=yes, type unknown.
- Roads: mostly residential and unclassified, no motorway or trunk.
- Hospitals are identified by amenity=hospital, only 1 building uses building=hospital.
- Edits from 2008 to 2026, with a strong spike in 2026
- Sanity checks: no duplicate versions, no roads with < 2 nodes, no building ways with < 4 nodes, all dates valid.
- Limitation: domain consistency for roads can’t be tested on this table (only the 17 listed values are kept), so it will use the snapshot.